In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q gliner transformers torch accelerate google-generativeai groq


## Step Description

1. **User Query**  
   The original input provided by the user.

2. **auto_discover_metadata()**  
   Automatically detects:
   - Domain of the query  
   - User intent  
   - Relevant Named Entity Recognition (NER) labels.

3. **stage_1_encoder()**  
   Performs:
   - Intent classification  
   - Named Entity Recognition to identify entities in the query.

4. **stage_2_classifier()**  
   Classifies extracted entities into different **privacy sensitivity categories**.

5. **stage_3_abstraction()**  
   Sanitizes the query by masking or abstracting sensitive information.

6. **finalize_and_repersonalize()**  
   - Sends sanitized query to the LLM  
   - Generates response  
   - Restores placeholders with appropriate contextual values.

7. **Final Response**  
   The privacy-preserving answer returned to the user.

In [ ]:
import json
import os
from gliner import GLiNER
from transformers import pipeline
from groq import Groq

# 1. Setup Clients and Local Models
# Using the new Llama 3.1 8B for fast, cost-effective tasks
# and Llama 3.3 70B for higher-level reasoning
MODELS = {
    "fast": "llama-3.1-8b-instant",
    "reasoning": "llama-3.3-70b-versatile"
}

try:
    from kaggle_secrets import UserSecretsClient
    client = Groq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))
except:
    client = Groq(api_key="YOUR_API_KEY")

# Load local NER model
ner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
# Local Zero-shot classifier (BART-large is still a great local choice)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

# --- STAGE 1: Automated Intent & Label Discovery ---
def auto_discover_metadata(query):
    prompt = f"""
    Analyze this user query: "{query}"
    1. Identify the general Domain (e.g., Healthcare, Finance, Engineering).
    2. List 3-5 possible specific Intents.
    3. List 5-8 entity types (labels) for NER (e.g., 'patient_name', 'location', 'medical_condition').
    
    Return ONLY a JSON object: 
    {{"domain": "string", "intents": [], "ner_labels": []}}
    """
    completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODELS["fast"],
        response_format={"type": "json_object"}
    )
    metadata = json.loads(completion.choices[0].message.content)
    print(f"\n--- [DEBUG] Auto-Discovered Metadata ---")
    print(json.dumps(metadata, indent=2))
    return metadata

# --- STAGE 2: NER & Intent Encoding (E) ---
def stage_1_encoder(query, metadata):
    # Intent Classification using dynamic labels
    intent_results = classifier(query, metadata['intents'])
    top_intent = intent_results['labels'][0]
    
    # NER using dynamic labels
    entities = ner_model.predict_entities(query, metadata['ner_labels'])
    
    print(f"\n--- [DEBUG] Stage 1: Encoder Output ---")
    print(f"Final Intent: {top_intent}")
    print(f"Detected Entities: {[f'{e['text']} ({e['label']})' for e in entities]}")
    return top_intent, entities

# --- STAGE 3: Impact-Based Token Classification (Tc) ---
def stage_2_classifier(query, intent, entities):
    # Extract literal text to ensure the LLM only uses what exists in the query
    entity_literals = [e['text'] for e in entities]
    
    prompt = f"""
    SYSTEM: You are a Privacy-Preserving Logic Engine. 
    TASK: Partition the 'Literal Entities' into THREE mutually exclusive JSON lists based on the 'Intent'.
    
    RULES:
    1. S_drop: Sensitive data (PII) that has ZERO impact on fulfilling the intent. (e.g., Names, addresses, or specific IDs). [cite: 31]
    2. S_abstract: Sensitive data that IS REQUIRED for a logical/correct answer. (e.g., A disease name in a medical query). [cite: 33]
    3. T_core: Non-sensitive data or general technical terms that must stay untouched. [cite: 35]

    FEW-SHOT EXAMPLES:
    Example 1 (Medical):
    - Intent: inquire_medication_safety
    - Entities: ["Rajesh Kumar", "Diabetes", "Ibuprofen"]
    - Result: {{"S_drop": ["Rajesh Kumar"], "S_abstract": ["Diabetes"], "T_core": ["Ibuprofen"]}}

    Example 2 (Finance):
    - Intent: dispute_transaction
    - Entities: ["Account 1234", "Netflix", "$50"]
    - Result: {{"S_drop": [], "S_abstract": ["Account 1234", "$50"], "T_core": ["Netflix"]}}

    CURRENT TASK:
    - Query: "{query}"
    - Intent: "{intent}"
    - Literal Entities: {entity_literals}

    Return ONLY a JSON object. Ensure every string in 'Literal Entities' is assigned to exactly one list.
    """

    completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODELS["fast"],
        # CRITICAL: Set temperature to 0 for deterministic classification
        temperature=0, 
        response_format={"type": "json_object"}
    )
    
    return completion.choices[0].message.content

# --- STAGE 4: Adaptive Privacy Abstraction (P) ---
def stage_3_abstraction(query, classification_json):
    classes = json.loads(classification_json)
    mapping_table = {}
    sanitized_query = query
    
    # helper function to ensure we get a string from LLM output
    def get_text(item):
        if isinstance(item, dict):
            return item.get("text", "")
        return str(item)

    # 1. Process S_drop (Delete)
    for item in classes.get("S_drop", []):
        token = get_text(item)
        if token:
            sanitized_query = sanitized_query.replace(token, "[REDACTED]")
        
    # 2. Process S_abstract (Map to Placeholder)
    for i, item in enumerate(classes.get("S_abstract", [])):
        token = get_text(item)
        if token:
            placeholder = f"[PROTECTED_VALUE_{i+1}]"
            mapping_table[placeholder] = token
            sanitized_query = sanitized_query.replace(token, placeholder)
        
    print(f"\n--- [DEBUG] Stage 3: Sanitized Query ---")
    print(f"Sanitized: {sanitized_query}")
    return sanitized_query, mapping_table

# --- STAGE 5: Generation (G) & Re-Personalization (R) ---
def finalize_and_repersonalize(sanitized_query, mapping_table):
    # Generation using the more powerful 70B model
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": sanitized_query}],
        model=MODELS["reasoning"],
    )
    raw_answer = response.choices[0].message.content
    print(f"\n--- [DEBUG] Stage 4: LLM Raw Output ---")
    print(raw_answer)
    
    # Re-Personalization
    final_answer = raw_answer
    for placeholder, real_value in mapping_table.items():
        final_answer = final_answer.replace(placeholder, real_value)
    
    return final_answer

def run_pipeline(query):
    print(f"INPUT: {query}\n" + "="*60)
    meta = auto_discover_metadata(query)
    intent, entities = stage_1_encoder(query, meta)
    classification = stage_2_classifier(query, intent, entities)
    print(f"\n--- [DEBUG] Stage 2: Token Classification ---\n{classification}")
    sanitized_q, mapping = stage_3_abstraction(query, classification)
    result = finalize_and_repersonalize(sanitized_q, mapping)
    print("\n" + "="*60 + "\nFINAL OUTPUT:\n" + result)

In [ ]:
# Example Execution
run_pipeline("I am Inarat in my last semester of btech. How to brush up on my java skills in last 3 months before placements?")

In [ ]:
# Example Execution
run_pipeline("I am Akash from IIIT Vadodara. My bank account 987654321 is showing a weird charge of $50 from Netflix. Help me file a dispute.")

In [ ]:
# Example Execution
run_pipeline("My name is Rajesh Kumar. I live at Flat 402, Mumbai. I lost my credit card 4567 1234 5678 9999 yesterday and I also have diabetes. Can I take ibuprofen safely?")

In [ ]:
test_query_1 = "My name is Pascal and I am struggling to write a compiler in Pascal. Also, I live in Paris, France."
# Expected: 'Pascal' (name) -> S_drop, 'Pascal' (language) -> T_core, 'Paris' -> S_drop.
run_pipeline(test_query_1)

In [ ]:
test_query_2 = "I have Type 1 Diabetes and I am currently taking Lisinopril. Can I eat a high-sugar grapefruit breakfast while on this medication?"
# Expected: 'Type 1 Diabetes' -> S_abstract, 'Lisinopril' -> T_core (or S_abstract), 'high-sugar' -> T_core.
run_pipeline(test_query_2)

In [ ]:
test_query_3 = "How do I calculate compound interest? My current balance is $54,320.12 in account 99887766 and my social security number is 000-11-2222."
# Expected: All numbers -> S_drop. The intent (calculate interest) only requires a formula, not the user's actual balance or SSN.
run_pipeline(test_query_3)

In [ ]:
test_query_4 = "Why is my Python script failing? I am using the 'requests' library to call 'https://api.stripe.com' with the key 'sk_live_51Mzj82L09abc123' and I am located in Berlin."
# Expected: 'requests' -> T_core, 'Berlin' -> S_drop, 'sk_live...' -> S_drop (or S_abstract if testing a 'credential' placeholder).
run_pipeline(test_query_4)

In [ ]:
test_query_5 ="Here is a KYC snippet: 'Customer Rajesh Kumar, residing at Flat 402, Sea View, Mumbai.' Can you write a welcome letter?"
run_pipeline(test_query_5)

## Benchmark Evaluation Flow

Benchmark  
  

In [ ]:
##Metadata testing and benchmarking starts from here

In [ ]:
"""
Builds the metadata cache by running auto_discover_metadata for every
sample across 20 runs. Fully resumable — safe to interrupt and re-run.

Features:
  - Controlled domain vocabulary in the prompt (fixes freeform domain fragmentation)
  - Uses domain and intent as ground truth from benchmark dataset
"""

import pandas as pd
import ast
import json
import numpy as np
import time
import os
from gliner import GLiNER
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from groq import Groq, RateLimitError
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("key2")
secret_value_1 = user_secrets.get_secret("key3")
secret_value_2 = user_secrets.get_secret("MY_KEY")

In [ ]:
############################################################
# CLIENTS & MODELS
############################################################
MODELS = {
    "fast":      "llama-3.1-8b-instant",
    "reasoning": "llama-3.3-70b-versatile"
}

try:
    client = Groq(api_key=secret_value_0)
except Exception as e:
    raise RuntimeError(f"Could not initialise Groq client: {e}")

ner_model   = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
classifier  = pipeline("zero-shot-classification",
                       model="facebook/bart-large-mnli", device=-1)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

############################################################
# CACHE FILES
############################################################
METADATA_CACHE_FILE   = "/kaggle/working/metadata_cache.json"
EXPERIMENT_CACHE_FILE = "/kaggle/working/metadata_experiment_cache.json"

def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f)

############################################################
# DATASET
############################################################
CSV_PATH = "/kaggle/input/datasets/inarathussain/dataset2/btp_privacy_benchmark_1000.csv"

df = pd.read_csv(CSV_PATH)
df = df[df["original_query"].notna() & (df["original_query"].str.strip() != "")]
df = df.reset_index(drop=True)

benchmark_dataset = [
    {
        "query":     row["original_query"],
        "gt_domain": str(row.get("domain", "")).strip(),
        "gt_intent": str(row.get("intent", "")).strip(),
    }
    for _, row in df.iterrows()
]

print(f"Loaded {len(benchmark_dataset)} samples.")
print(f"Ground-truth domains : {df['domain'].nunique()} unique")
print(f"Ground-truth intents : {df['intent'].nunique()} unique\n")

In [ ]:
############################################################
# CONTROLLED VOCABULARY
# Derived from the actual domain column in the CSV so the
# model picks from labels that exist in the ground truth.
############################################################
DOMAIN_VOCABULARY = sorted(df["domain"].dropna().unique().tolist())
DOMAIN_VOCAB_STR  = ", ".join(DOMAIN_VOCABULARY)

############################################################
# auto_discover_metadata  (with controlled domain vocabulary)
############################################################
def auto_discover_metadata(query: str) -> dict:
    prompt = f"""
    Analyze this user query: "{query}"

    1. Identify the domain. You MUST choose exactly one domain from this list:
       {DOMAIN_VOCAB_STR}
       Output the domain exactly as written above — do not invent new domain names.

    2. List 3-5 possible specific Intents for this query (free text, be specific).

    3. List 5-8 entity types (labels) for NER that are relevant to this query
       (e.g., 'patient_name', 'location', 'medical_condition').

    Return ONLY valid JSON with this EXACT schema:
    {{
      "domain": "string",
      "intents": ["intent1", "intent2", "intent3"],
      "ner_labels": ["label1", "label2", "label3"]
    }}
    Rules:
    - domain MUST be one of the allowed values listed above
    - intents MUST be a list of strings
    - ner_labels MUST be a list of strings
    - No nested objects inside lists
    - Output strictly valid JSON
    """
    while True:
        try:
            completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=MODELS["fast"],
                response_format={"type": "json_object"}
            )
            result = json.loads(completion.choices[0].message.content)

            # Validate domain is from vocabulary — snap to closest if not
            predicted_domain = result.get("domain", "").strip()
            if predicted_domain not in DOMAIN_VOCABULARY:
                matches = [d for d in DOMAIN_VOCABULARY
                           if predicted_domain.lower() in d.lower()
                           or d.lower() in predicted_domain.lower()]
                result["domain"] = matches[0] if matches else DOMAIN_VOCABULARY[0]

            return result

        except RateLimitError:
            print("  [auto_discover] Rate limited — waiting 60 s...")
            time.sleep(60)
        except Exception as e:
            print(f"  [auto_discover] Error: {e} — retrying in 10 s...")
            time.sleep(10)

############################################################
# CACHED METADATA FETCH  (per-run keys for stability measurement)
############################################################
def get_metadata_cached(metadata_cache: dict,
                        run_num: int,
                        sample_idx: int,
                        query: str) -> tuple[dict, bool]:
    key = f"sample_{sample_idx}_run_{run_num}"

    if key in metadata_cache:
        return json.loads(metadata_cache[key]), True

    metadata = auto_discover_metadata(query)
    metadata_cache[key] = json.dumps(metadata)
    save_json(METADATA_CACHE_FILE, metadata_cache)

    return metadata, False

In [ ]:
############################################################
# SINGLE RUN
############################################################
def run_single(metadata_cache: dict,
               exp_cache: dict,
               run_num: int,
               start_sample: int = 0) -> list[dict]:

    run_key = f"run_{run_num}_results"
    results = exp_cache.get(run_key, [None] * len(benchmark_dataset))

    for i, sample in enumerate(benchmark_dataset):
        if results[i] is not None:
            continue

        try:
            meta, was_cached = get_metadata_cached(
                metadata_cache, run_num, i, sample["query"]
            )
        except Exception as e:
            exp_cache["progress"] = {"run": run_num, "sample": i}
            exp_cache[run_key]    = results
            save_json(EXPERIMENT_CACHE_FILE, exp_cache)
            print(f"\n⚠️  Error at run {run_num}, sample {i}: {e}")
            print("   Progress saved — re-run the cell to resume.")
            raise

        results[i] = meta

        exp_cache[run_key]    = results
        exp_cache["progress"] = {"run": run_num, "sample": i + 1}
        save_json(EXPERIMENT_CACHE_FILE, exp_cache)

        if (i + 1) % 100 == 0:
            cached_count = sum(
                1 for j in range(i + 1)
                if f"sample_{j}_run_{run_num}" in metadata_cache
            )
            print(f"    [Run {run_num:02d}] {i + 1}/{len(benchmark_dataset)} samples "
                  f"(cached: {cached_count}, live: {i + 1 - cached_count})")

    return results

############################################################
# 20-RUN EXPERIMENT
############################################################
def run_metadata_experiment(n_runs: int = 5) -> None:
    metadata_cache = load_json(METADATA_CACHE_FILE)
    exp_cache      = load_json(EXPERIMENT_CACHE_FILE)

    progress      = exp_cache.get("progress", {"run": 1, "sample": 0})
    resume_run    = progress["run"]
    resume_sample = progress["sample"]

    all_run_metadata = exp_cache.get("all_run_metadata", [])
    completed_runs   = len(all_run_metadata)

    print(f"Metadata Cache-Building Experiment")
    print(f"  {n_runs} runs × {len(benchmark_dataset)} samples")
    print(f"  Domain vocabulary : {DOMAIN_VOCAB_STR}\n")

    if completed_runs or resume_sample:
        print(f"▶  Resuming from Run {resume_run}, Sample {resume_sample} "
              f"({completed_runs} runs fully completed)\n")

    for run in range(resume_run, n_runs + 1):
        start_sample = resume_sample if run == resume_run else 0

        print(f"--- Run {run:02d}/{n_runs}  (start sample: {start_sample}) ---")
        run_results = run_single(metadata_cache, exp_cache, run, start_sample)

        if len(all_run_metadata) < run:
            all_run_metadata.append(run_results)

        exp_cache["all_run_metadata"] = all_run_metadata
        exp_cache["progress"]         = {"run": run + 1, "sample": 0}
        save_json(EXPERIMENT_CACHE_FILE, exp_cache)

        print(f"  ✅ Run {run:02d} complete\n")

    print(f"{'='*55}")
    print(f"  All {n_runs} runs complete.")
    print(f"  metadata_cache.json            → {len(load_json(METADATA_CACHE_FILE))} entries")
    print(f"  metadata_experiment_cache.json → {n_runs} runs stored")
    print(f"  Ready to run:")
    print(f"    → domain_intent_evaluation.py")
    print(f"    → domain_stability_analysis.py")
    print(f"{'='*55}\n")

In [ ]:
############################################################
# ENTRY POINT
############################################################
run_metadata_experiment(n_runs=5)

In [ ]:
"""
domain_intent_evaluation.py
────────────────────────────
Evaluates auto_discover_metadata against ground-truth domain and intent
columns from the benchmark CSV.

For each of the 20 runs independently:
  Domain:
    - Exact match rate         (predicted == ground truth, case-insensitive)
    - Semantic similarity      (cosine via all-MiniLM-L6-v2)
  Intent:
    - Exact match              (ground truth intent in any predicted intent, case-insensitive)
    - Semantic similarity      (max cosine between GT intent and any predicted intent)

Aggregates mean ± std across all 20 runs and breaks down by ground-truth domain.
"""

import json
import os
import ast
import pandas as pd
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer

############################################################
# CONFIG
############################################################
CSV_PATH          = "/kaggle/input/datasets/inarathussain/dataset2/btp_privacy_benchmark_1000.csv"
EXPERIMENT_CACHE  = "/kaggle/working/metadata_experiment_cache.json"
REPORT_FILE       = "/kaggle/working/domain_intent_eval_report.json"
MIN_SAMPLES       = 10     # minimum samples for per-domain breakdown

############################################################
# LOAD CACHE
############################################################
print("Loading experiment cache...")
if not os.path.exists(EXPERIMENT_CACHE):
    raise FileNotFoundError(
        f"Cache not found: {EXPERIMENT_CACHE}\n"
        "Run the 20-run benchmark first."
    )

with open(EXPERIMENT_CACHE) as f:
    exp_cache = json.load(f)

all_run_metadata = exp_cache.get("all_run_metadata", [])
if not all_run_metadata:
    raise ValueError("all_run_metadata is empty — benchmark must complete first.")

n_runs    = len(all_run_metadata)
n_samples = len(all_run_metadata[0])
print(f"  {n_runs} runs × {n_samples} samples loaded.\n")

############################################################
# LOAD GROUND TRUTH
############################################################
print("Loading ground truth from CSV...")
df = pd.read_csv(CSV_PATH)

# Filter to non-empty queries (same logic as benchmark script)
df = df[df["original_query"].notna() & (df["original_query"].str.strip() != "")]
df = df.reset_index(drop=True)

assert len(df) == n_samples, (
    f"CSV has {len(df)} valid rows but cache has {n_samples} samples — mismatch!"
)

gt_domains  = df["domain"].str.lower().str.strip().tolist()
gt_intents  = df["intent"].str.lower().str.strip().tolist()

print(f"  Ground-truth domains  : {df['domain'].nunique()} unique")
print(f"  Ground-truth intents  : {df['intent'].nunique()} unique\n")

############################################################
# EMBEDDING MODEL
############################################################
print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("  Ready.\n")

############################################################
# HELPERS
############################################################
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two already-normalised vectors."""
    return float(a @ b)


def domain_scores(predicted: str, ground_truth: str) -> tuple[int, float]:
    """
    Returns (exact_match, semantic_sim) for a single domain prediction.
    exact_match : 1 if strings match exactly (case-insensitive), else 0
    semantic_sim: cosine similarity between embeddings
    """
    pred_clean = predicted.lower().strip()
    gt_clean   = ground_truth.lower().strip()

    exact = int(pred_clean == gt_clean)

    embs = embed_model.encode([pred_clean, gt_clean], normalize_embeddings=True)
    sem  = cosine(embs[0], embs[1])

    return exact, sem


def intent_scores(predicted_list: list, ground_truth: str) -> tuple[int, float]:
    """
    Returns (exact_match, semantic_sim) for intent evaluation.

    exact_match : 1 if ground truth intent appears (exactly) in any predicted intent
    semantic_sim: max cosine similarity between GT intent embedding and
                  any predicted intent embedding (best-match)
    """
    gt_clean   = ground_truth.lower().strip()
    flat = []
    for p in predicted_list:
        if isinstance(p, list):
            flat.extend(p)
        else:
            flat.append(p)
    pred_clean = [p.lower().strip() for p in flat if p and isinstance(p, str)]

    if not pred_clean:
        return 0, 0.0

    # Exact match — check if GT intent is a substring of any predicted intent
    # (handles cases like GT="check_medication_interaction",
    #  predicted="check medication interaction")
    gt_normalised = gt_clean.replace("_", " ")
    exact = int(any(
        gt_normalised in p.replace("_", " ") or p.replace("_", " ") in gt_normalised
        for p in pred_clean
    ))

    # Semantic — embed GT + all predictions, find max sim
    all_texts = [gt_clean] + pred_clean
    embs      = embed_model.encode(all_texts, normalize_embeddings=True)
    gt_emb    = embs[0]
    pred_embs = embs[1:]
    sims      = [cosine(gt_emb, p) for p in pred_embs]
    sem       = float(max(sims))

    return exact, sem

############################################################
# EVALUATION LOOP — across all runs independently
############################################################
print("Evaluating domain and intent across all runs...\n")

# Shape: [run_idx][sample_idx] → {"domain_exact", "domain_sem", "intent_exact", "intent_sem"}
all_scores = []

for r in range(n_runs):
    run_scores = []
    for s in range(n_samples):
        meta       = all_run_metadata[r][s] or {}
        pred_domain  = meta.get("domain", "")
        pred_intents = meta.get("intents", [])

        d_exact, d_sem = domain_scores(pred_domain, gt_domains[s])
        i_exact, i_sem = intent_scores(pred_intents, gt_intents[s])

        run_scores.append({
            "domain_exact": d_exact,
            "domain_sem":   d_sem,
            "intent_exact": i_exact,
            "intent_sem":   i_sem,
            "gt_domain":    gt_domains[s],
            "gt_intent":    gt_intents[s],
        })

    all_scores.append(run_scores)

    # Per-run summary
    d_ex  = np.mean([s["domain_exact"] for s in run_scores])
    d_sem = np.mean([s["domain_sem"]   for s in run_scores])
    i_ex  = np.mean([s["intent_exact"] for s in run_scores])
    i_sem = np.mean([s["intent_sem"]   for s in run_scores])
    print(f"  Run {r+1:02d}/{n_runs}  |  "
          f"Domain exact={d_ex:.3f}  sem={d_sem:.3f}  |  "
          f"Intent exact={i_ex:.3f}  sem={i_sem:.3f}")

############################################################
# AGGREGATE ACROSS RUNS
############################################################
def aggregate(metric: str) -> tuple[float, float]:
    """Mean and std of a metric across all runs."""
    per_run = [np.mean([s[metric] for s in run]) for run in all_scores]
    return float(np.mean(per_run)), float(np.std(per_run))

d_exact_mean,  d_exact_std  = aggregate("domain_exact")
d_sem_mean,    d_sem_std    = aggregate("domain_sem")
i_exact_mean,  i_exact_std  = aggregate("intent_exact")
i_sem_mean,    i_sem_std    = aggregate("intent_sem")

print(f"\n{'='*60}")
print(f"  Aggregate Results  ({n_runs} runs)")
print(f"{'='*60}")
print(f"  {'Metric':<35} {'Mean':>7}  {'Std':>7}")
print(f"  {'-'*52}")
print(f"  {'Domain Exact Match Rate':<35} {d_exact_mean:>7.4f}  {d_exact_std:>7.4f}")
print(f"  {'Domain Semantic Similarity':<35} {d_sem_mean:>7.4f}  {d_sem_std:>7.4f}")
print(f"  {'Intent Exact Match Rate':<35} {i_exact_mean:>7.4f}  {i_exact_std:>7.4f}")
print(f"  {'Intent Semantic Similarity':<35} {i_sem_mean:>7.4f}  {i_sem_std:>7.4f}")
print(f"{'='*60}\n")

############################################################
# PER-DOMAIN BREAKDOWN  (using ground-truth domain labels)
############################################################
print(f"Per-domain breakdown (min {MIN_SAMPLES} samples)...\n")

# Bucket sample indices by ground-truth domain
domain_indices = defaultdict(list)
for s in range(n_samples):
    domain_indices[gt_domains[s]].append(s)

per_domain_report = []

for domain, indices in sorted(domain_indices.items()):
    if len(indices) < MIN_SAMPLES:
        continue

    # Average each metric across all runs for samples in this domain
    d_ex_vals, d_sem_vals, i_ex_vals, i_sem_vals = [], [], [], []
    for r in range(n_runs):
        d_ex_vals.append( np.mean([all_scores[r][s]["domain_exact"] for s in indices]))
        d_sem_vals.append(np.mean([all_scores[r][s]["domain_sem"]   for s in indices]))
        i_ex_vals.append( np.mean([all_scores[r][s]["intent_exact"] for s in indices]))
        i_sem_vals.append(np.mean([all_scores[r][s]["intent_sem"]   for s in indices]))

    per_domain_report.append({
        "domain":              domain,
        "sample_count":        len(indices),
        "domain_exact_mean":   round(float(np.mean(d_ex_vals)),  4),
        "domain_exact_std":    round(float(np.std(d_ex_vals)),   4),
        "domain_sem_mean":     round(float(np.mean(d_sem_vals)), 4),
        "domain_sem_std":      round(float(np.std(d_sem_vals)),  4),
        "intent_exact_mean":   round(float(np.mean(i_ex_vals)),  4),
        "intent_exact_std":    round(float(np.std(i_ex_vals)),   4),
        "intent_sem_mean":     round(float(np.mean(i_sem_vals)), 4),
        "intent_sem_std":      round(float(np.std(i_sem_vals)),  4),
    })

# Sort by intent semantic similarity descending
per_domain_report.sort(key=lambda x: x["intent_sem_mean"], reverse=True)

col = 25
print(f"  {'Domain':<{col}} {'N':>4}  "
      f"{'DomEx':>6}  {'DomSem':>6}  {'IntEx':>6}  {'IntSem':>6}")
print(f"  {'-'*65}")
for r in per_domain_report:
    print(f"  {r['domain']:<{col}} {r['sample_count']:>4}  "
          f"{r['domain_exact_mean']:>6.3f}  "
          f"{r['domain_sem_mean']:>6.3f}  "
          f"{r['intent_exact_mean']:>6.3f}  "
          f"{r['intent_sem_mean']:>6.3f}")

print(f"\n  Legend:")
print(f"    DomEx  = Domain exact match rate  (predicted string == GT string)")
print(f"    DomSem = Domain semantic similarity (cosine to GT)")
print(f"    IntEx  = Intent exact match rate  (GT intent found in predicted list)")
print(f"    IntSem = Intent semantic similarity (best cosine match to GT intent)")

############################################################
# SAVE REPORT
############################################################
report = {
    "n_runs":    n_runs,
    "n_samples": n_samples,
    "aggregate": {
        "domain_exact_mean":  d_exact_mean,
        "domain_exact_std":   d_exact_std,
        "domain_sem_mean":    d_sem_mean,
        "domain_sem_std":     d_sem_std,
        "intent_exact_mean":  i_exact_mean,
        "intent_exact_std":   i_exact_std,
        "intent_sem_mean":    i_sem_mean,
        "intent_sem_std":     i_sem_std,
    },
    "per_domain": per_domain_report,
}

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print(f"\n  Full report saved → {REPORT_FILE}")

In [ ]:
"""
cache_gliner_predictions.py
────────────────────────────
Step 1 of 2 — Run ONCE.

Reads the experiment cache (metadata_experiment_cache.json) which contains
the NER label lists produced by auto_discover_metadata, then runs GLiNER
once per unique (query, ner_labels) pair and writes every prediction to
a new cache file: gliner_predictions_cache.json.

Subsequent evaluation scripts (Step 2) load this cache and never touch
GLiNER again, making all metric calculations fast and fully reproducible.

Output schema
─────────────
{
  "meta": {
    "n_runs":    <int>,
    "n_samples": <int>,
    "model":     "urchade/gliner_multi-v2.1",
    "created":   "<ISO timestamp>"
  },
  "predictions": [          ← outer list: one entry per sample
    [                       ← inner list: one entry per run
      {
        "query":      <str>,
        "ner_labels": [<str>, ...],
        "predicted":  [<str>, ...]   ← normalised token strings from GLiNER
      },
      ...
    ],
    ...
  ]
}
"""

import json
import os
import ast
import hashlib
import datetime
import pandas as pd
from gliner import GLiNER

############################################################
# CONFIG  — adjust paths as needed
############################################################
CSV_PATH         = "/kaggle/input/datasets/inarathussain/dataset2/btp_privacy_benchmark_1000.csv"
EXPERIMENT_CACHE = "/kaggle/working/metadata_experiment_cache.json"
PREDICTIONS_CACHE = "/kaggle/working/gliner_predictions_cache.json"

############################################################
# HELPERS
############################################################

def normalize(token: str) -> str:
    return str(token).lower().strip()


def cache_key(query: str, ner_labels: list) -> str:
    """Deterministic key so identical (query, labels) pairs share one prediction."""
    payload = json.dumps({"q": query, "l": sorted(ner_labels)}, sort_keys=True)
    return hashlib.md5(payload.encode()).hexdigest()


def safe_parse(val):
    try:
        return ast.literal_eval(str(val))
    except Exception:
        return []

############################################################
# LOAD EXPERIMENT CACHE
############################################################
print("Loading experiment cache...")
with open(EXPERIMENT_CACHE) as f:
    exp_cache = json.load(f)

all_run_metadata = exp_cache["all_run_metadata"]
n_runs   = len(all_run_metadata)
n_samples = len(all_run_metadata[0])
print(f"  {n_runs} runs × {n_samples} samples\n")

############################################################
# LOAD CSV — queries only (ground truth not needed here)
############################################################
print("Loading queries from CSV...")
df = pd.read_csv(CSV_PATH)
df = df[df["original_query"].notna() & (df["original_query"].str.strip() != "")]
df = df.reset_index(drop=True)

assert len(df) == n_samples, (
    f"CSV rows ({len(df)}) != cache samples ({n_samples})"
)
queries = df["original_query"].tolist()
print(f"  {n_samples} queries loaded.\n")

############################################################
# LOAD GLiNER
############################################################
MODEL_NAME = "urchade/gliner_multi-v2.1"
print(f"Loading GLiNER model: {MODEL_NAME}...")
ner_model = GLiNER.from_pretrained(MODEL_NAME)
print("  Model ready.\n")

############################################################
# DEDUPLICATED INFERENCE
# We collect all unique (query, ner_labels) pairs first,
# run GLiNER once per unique pair, then build the full matrix.
############################################################
print("Collecting unique (query, ner_labels) pairs...")

# unique_pairs: key → (query, ner_labels)
unique_pairs: dict[str, tuple[str, list]] = {}

for r in range(n_runs):
    for s in range(n_samples):
        meta       = all_run_metadata[r][s] or {}
        ner_labels = meta.get("ner_labels", [])
        if not ner_labels:
            continue
        key = cache_key(queries[s], ner_labels)
        if key not in unique_pairs:
            unique_pairs[key] = (queries[s], ner_labels)

total_unique = len(unique_pairs)
print(f"  {total_unique} unique (query, ner_labels) pairs to run through GLiNER.\n")

# Run GLiNER — one call per unique pair
print("Running GLiNER predictions (this may take a while)...")
dedup_results: dict[str, list[str]] = {}   # key → list of normalised predicted tokens

for i, (key, (query, ner_labels)) in enumerate(unique_pairs.items(), 1):
    entities = ner_model.predict_entities(query, ner_labels)
    dedup_results[key] = [normalize(e["text"]) for e in entities]

    if i % 100 == 0 or i == total_unique:
        print(f"  Processed {i}/{total_unique} unique pairs...")

print(f"\n  Inference complete.\n")

############################################################
# BUILD FULL PREDICTION MATRIX  [n_samples][n_runs]
############################################################
print("Building prediction matrix...")

# predictions[s][r] = dict with query, ner_labels, predicted list (or None if skipped)
predictions = []

for s in range(n_samples):
    sample_preds = []
    for r in range(n_runs):
        meta       = all_run_metadata[r][s] or {}
        ner_labels = meta.get("ner_labels", [])
        query      = queries[s]

        if not ner_labels:
            # No NER labels cached for this run/sample → mark as skipped
            sample_preds.append({
                "query":      query,
                "ner_labels": [],
                "predicted":  None      # None signals "skipped"
            })
        else:
            key = cache_key(query, ner_labels)
            sample_preds.append({
                "query":      query,
                "ner_labels": ner_labels,
                "predicted":  dedup_results[key]
            })
    predictions.append(sample_preds)

############################################################
# SAVE CACHE
############################################################
output = {
    "meta": {
        "n_runs":    n_runs,
        "n_samples": n_samples,
        "model":     MODEL_NAME,
        "created":   datetime.datetime.utcnow().isoformat() + "Z",
    },
    "predictions": predictions,
}

with open(PREDICTIONS_CACHE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Prediction cache saved → {PREDICTIONS_CACHE}")
print(f"  Rows (samples): {n_samples}")
print(f"  Cols (runs):    {n_runs}")
print(f"  Unique GLiNER calls made: {total_unique}")
skipped_total = sum(
    1
    for s in range(n_samples)
    for r in range(n_runs)
    if predictions[s][r]["predicted"] is None
)
print(f"  Skipped (no ner_labels): {skipped_total} / {n_runs * n_samples}")
print("\nDone. Run evaluate_all_metrics.py next.")

In [ ]:
"""
evaluate_all_metrics.py
────────────────────────
Step 2 of 2 — Run as many times as needed, zero GLiNER calls.

Loads gliner_predictions_cache.json (written by cache_gliner_predictions.py)
and the ground-truth CSV, then computes five metrics in TWO averaging modes:

  MICRO  — pool all token bits across every sample in a run, score once.
            Weights each token equally; dominated by high-token-count samples.

  MACRO  — score each sample independently, then average across samples.
            Weights each sample equally regardless of token count.

Both modes are reported at every level: per-run console line, aggregate
summary table, per-domain breakdown, and the JSON report.

┌─────────────────────────────────────────────────────────────────────────┐
│  Metric               │ Abbr  │ Range  │ Averaging modes                │
├───────────────────────┼───────┼────────┼────────────────────────────────┤
│  Token Recall         │ rec   │ [0,1]  │ micro + macro                  │
│  Token Precision      │ pre   │ [0,1]  │ micro + macro                  │
│  Token F1             │ f1    │ [0,1]  │ micro + macro                  │
│  Intent Preservation  │ ips   │ [0,1]  │ macro only (inherently scalar) │
│  Privacy Leakage      │ plr   │ [0,1]  │ macro only (inherently scalar) │
└─────────────────────────────────────────────────────────────────────────┘

Metric definitions
──────────────────
Let GT  = set of ground-truth sensitive tokens for a sample
    P   = set of tokens predicted sensitive by GLiNER

  recall    = |GT ∩ P| / |GT|                          (0 when |GT|=0)
  precision = |GT ∩ P| / |P|                           (0 when |P|=0)
  f1        = 2·pre·rec / (pre+rec)                    (0 when both 0)

  IPS (Intent Preservation Score)
    non_gt = query tokens NOT in GT
    IPS    = 1 − |FP ∩ non_gt| / |non_gt|
    Skipped when non_gt is empty.

  PLR (Privacy Leakage Ratio)
    PLR = |GT \ P| / |GT|  =  1 − recall
    Skipped when GT is empty.

Skip policy
───────────
  A (sample, run) pair is globally skipped when:
    • predicted is None  (no NER labels were cached)
    • GT and P are both empty

  IPS additionally skips when there are no non-sensitive query tokens.
  PLR additionally skips when GT is empty.

Output
──────
  all_metrics_report.json  — full structured report (micro + macro)
  Console                  — human-readable tables
"""

import json
import re
import ast
import datetime
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score

############################################################
# CONFIG
############################################################
CSV_PATH          = "/kaggle/input/datasets/inarathussain/dataset2/btp_privacy_benchmark_1000.csv"
PREDICTIONS_CACHE = "/kaggle/working/gliner_predictions_cache.json"
REPORT_FILE       = "/kaggle/working/all_metrics_report.json"
MIN_SAMPLES       = 10   # min evaluable samples for per-domain table

############################################################
# HELPERS
############################################################

def normalize(token: str) -> str:
    return str(token).lower().strip()


def safe_parse(val):
    try:
        return ast.literal_eval(str(val))
    except Exception:
        return []


def simple_tokenize(text: str) -> set:
    """Lightweight word tokenizer used for IPS non-sensitive token enumeration."""
    return set(re.findall(r"\b\w+(?:'\w+)?\b", text.lower()))


def agg_stats(values: list) -> dict:
    """mean/std/min/max over a list, ignoring None and NaN."""
    v = [x for x in values if x is not None and not np.isnan(float(x))]
    if not v:
        nan = float("nan")
        return {"mean": nan, "std": nan, "min": nan, "max": nan, "n": 0}
    return {
        "mean": round(float(np.mean(v)), 4),
        "std":  round(float(np.std(v)),  4),
        "min":  round(float(np.min(v)),  4),
        "max":  round(float(np.max(v)),  4),
        "n":    len(v),
    }


def micro_scores(y_true, y_pred):
    """Pooled (micro) recall, precision, F1 over accumulated binary vectors."""
    if not y_true:
        nan = float("nan")
        return nan, nan, nan
    rec = float(recall_score(y_true, y_pred, zero_division=0))
    pre = float(precision_score(y_true, y_pred, zero_division=0))
    f1  = float(f1_score(y_true, y_pred, zero_division=0))
    return rec, pre, f1


############################################################
# PER-SAMPLE METRIC COMPUTATION
############################################################

def compute_sample_metrics(query, predicted, gt_tokens):
    """
    Returns per-sample scalars for macro averaging AND
    the binary token vectors needed for micro accumulation.

    Return dict keys:
      macro   → {recall, precision, f1, ips, plr}  (float | None each)
      y_true  → list[int]   (empty when globally skipped)
      y_pred  → list[int]   (empty when globally skipped)
      skipped → bool
    """
    MACRO_KEYS = ("recall", "precision", "f1", "ips", "plr")
    gt_set = set(normalize(t) for t in gt_tokens)

    # ── Global skip ─────────────────────────────────────────
    if predicted is None:
        return {"macro": {k: None for k in MACRO_KEYS},
                "y_true": [], "y_pred": [], "skipped": True}

    pred_set = set(predicted)          # already normalised in cache
    all_tok  = gt_set | pred_set

    if not all_tok:
        return {"macro": {k: None for k in MACRO_KEYS},
                "y_true": [], "y_pred": [], "skipped": True}

    # ── Binary vectors ───────────────────────────────────────
    y_true = [1 if t in gt_set   else 0 for t in all_tok]
    y_pred = [1 if t in pred_set else 0 for t in all_tok]

    macro = {}
    macro["recall"]    = float(recall_score(y_true, y_pred, zero_division=0))
    macro["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    macro["f1"]        = float(f1_score(y_true, y_pred, zero_division=0))

    # IPS
    non_gt = simple_tokenize(query) - gt_set
    if non_gt:
        fp_in_query  = (pred_set - gt_set) & non_gt
        macro["ips"] = 1.0 - len(fp_in_query) / len(non_gt)
    else:
        macro["ips"] = None

    # PLR
    macro["plr"] = (len(gt_set - pred_set) / len(gt_set)) if gt_set else None

    return {"macro": macro, "y_true": y_true, "y_pred": y_pred, "skipped": False}


############################################################
# LOAD CACHES
############################################################
print("Loading prediction cache...")
with open(PREDICTIONS_CACHE) as f:
    pred_cache = json.load(f)

meta_info   = pred_cache["meta"]
predictions = pred_cache["predictions"]
n_runs      = meta_info["n_runs"]
n_samples   = meta_info["n_samples"]
print(f"  Model  : {meta_info['model']}")
print(f"  Created: {meta_info['created']}")
print(f"  {n_runs} runs x {n_samples} samples\n")

print("Loading ground truth CSV...")
df = pd.read_csv(CSV_PATH)
df = df[df["original_query"].notna() & (df["original_query"].str.strip() != "")]
df = df.reset_index(drop=True)
assert len(df) == n_samples, f"CSV rows ({len(df)}) != cache samples ({n_samples})"

queries      = df["original_query"].tolist()
gt_domains   = df["domain"].str.lower().str.strip().tolist()
gt_sensitive = [
    list(set(safe_parse(row["expected_S_drop"]) +
             safe_parse(row["expected_S_abstract"])))
    for _, row in df.iterrows()
]
print("  Ground truth loaded.\n")

############################################################
# MAIN EVALUATION LOOP
############################################################
print("Computing metrics across all runs...\n")

MACRO_METRICS = ("recall", "precision", "f1", "ips", "plr")

# sample_run_macro[s][r] = macro-dict | None  (None = globally skipped)
sample_run_macro = [[None] * n_runs for _ in range(n_samples)]

# Per-run accumulators
run_micro      = []   # run_micro[r]  = {rec, pre, f1}
run_macro_mean = []   # run_macro_mean[r] = {metric: mean_over_samples}
skipped_counts = []

for r in range(n_runs):
    y_true_run, y_pred_run = [], []
    macro_accum = {m: [] for m in MACRO_METRICS}
    skipped = 0

    for s in range(n_samples):
        result = compute_sample_metrics(
            queries[s],
            predictions[s][r]["predicted"],
            gt_sensitive[s],
        )

        if result["skipped"]:
            skipped += 1
            continue

        sample_run_macro[s][r] = result["macro"]

        # Micro: extend pooled vectors
        y_true_run.extend(result["y_true"])
        y_pred_run.extend(result["y_pred"])

        # Macro: collect per-sample scalars
        for m in MACRO_METRICS:
            v = result["macro"][m]
            if v is not None:
                macro_accum[m].append(v)

    skipped_counts.append(skipped)

    # Micro scores for this run
    rec_mi, pre_mi, f1_mi = micro_scores(y_true_run, y_pred_run)
    run_micro.append({"recall": rec_mi, "precision": pre_mi, "f1": f1_mi,
                      "_y_true": y_true_run, "_y_pred": y_pred_run})

    # Macro means for this run
    rm = {m: float(np.mean(macro_accum[m])) if macro_accum[m] else float("nan")
          for m in MACRO_METRICS}
    run_macro_mean.append(rm)

    # Console
    if y_true_run:
        print(
            f"  Run {r+1:02d}/{n_runs}  "
            f"[mi] Rec={rec_mi:.4f} Pre={pre_mi:.4f} F1={f1_mi:.4f}  "
            f"[ma] Rec={rm['recall']:.4f} Pre={rm['precision']:.4f} "
            f"F1={rm['f1']:.4f} IPS={rm['ips']:.4f} PLR={rm['plr']:.4f}  "
            f"(skip {skipped}/{n_samples})"
        )
    else:
        print(f"  Run {r+1:02d}/{n_runs}  -> all samples skipped")

############################################################
# AGGREGATE ACROSS RUNS
############################################################

agg_micro = {
    m: agg_stats([run_micro[r][m] for r in range(n_runs)])
    for m in ("recall", "precision", "f1")
}
agg_macro = {
    m: agg_stats([run_macro_mean[r][m] for r in range(n_runs)])
    for m in MACRO_METRICS
}

SEP = "=" * 72
print(f"\n{SEP}")
print(f"  Aggregate  ({n_runs} runs x {n_samples} samples  |  "
      f"avg skipped: {np.mean(skipped_counts):.1f}/{n_samples})")
print(f"{SEP}")

print(f"\n  -- MICRO  (token bits pooled per run, averaged across runs) --")
print(f"  {'Metric':<12}  {'Mean':>7}  {'Std':>7}  {'Min':>7}  {'Max':>7}  {'Runs':>5}")
print(f"  {'-'*52}")
for m in ("recall", "precision", "f1"):
    a = agg_micro[m]
    print(f"  {m:<12}  {a['mean']:>7.4f}  {a['std']:>7.4f}  "
          f"{a['min']:>7.4f}  {a['max']:>7.4f}  {a['n']:>5}")

print(f"\n  -- MACRO  (per-sample score -> mean per run -> mean across runs) --")
print(f"  {'Metric':<12}  {'Mean':>7}  {'Std':>7}  {'Min':>7}  {'Max':>7}  {'Runs':>5}")
print(f"  {'-'*52}")
for m in MACRO_METRICS:
    a = agg_macro[m]
    print(f"  {m:<12}  {a['mean']:>7.4f}  {a['std']:>7.4f}  "
          f"{a['min']:>7.4f}  {a['max']:>7.4f}  {a['n']:>5}")

print(f"\n{SEP}\n")

############################################################
# PER-DOMAIN BREAKDOWN  (both micro and macro)
############################################################
print(f"Per-domain breakdown (min {MIN_SAMPLES} evaluable samples)...\n")

domain_indices = defaultdict(list)
for s, dom in enumerate(gt_domains):
    domain_indices[dom].append(s)

per_domain_report = []

for domain, indices in sorted(domain_indices.items()):
    evaluable = [s for s in indices
                 if any(sample_run_macro[s][r] is not None for r in range(n_runs))]
    if len(evaluable) < MIN_SAMPLES:
        continue

    d_micro_runs = {m: [] for m in ("recall", "precision", "f1")}
    d_macro_runs = {m: [] for m in MACRO_METRICS}

    for r in range(n_runs):
        # Rebuild micro vectors for this domain x run
        y_true_d, y_pred_d = [], []
        macro_vals = {m: [] for m in MACRO_METRICS}

        for s in evaluable:
            m_dict = sample_run_macro[s][r]
            if m_dict is None:
                continue
            # Recompute binary vectors from cache for micro
            gt_set   = set(normalize(t) for t in gt_sensitive[s])
            pred_set = set(predictions[s][r]["predicted"]) \
                       if predictions[s][r]["predicted"] else set()
            all_tok  = gt_set | pred_set
            for tok in all_tok:
                y_true_d.append(1 if tok in gt_set   else 0)
                y_pred_d.append(1 if tok in pred_set else 0)
            for m in MACRO_METRICS:
                v = m_dict[m]
                if v is not None:
                    macro_vals[m].append(v)

        if y_true_d:
            rec_d, pre_d, f1_d = micro_scores(y_true_d, y_pred_d)
            d_micro_runs["recall"].append(rec_d)
            d_micro_runs["precision"].append(pre_d)
            d_micro_runs["f1"].append(f1_d)

        for m in MACRO_METRICS:
            if macro_vals[m]:
                d_macro_runs[m].append(float(np.mean(macro_vals[m])))

    d_agg_micro = {m: agg_stats(d_micro_runs[m]) for m in ("recall", "precision", "f1")}
    d_agg_macro = {m: agg_stats(d_macro_runs[m]) for m in MACRO_METRICS}

    per_domain_report.append({
        "domain":            domain,
        "sample_count":      len(indices),
        "evaluable_samples": len(evaluable),
        "micro": {m: d_agg_micro[m] for m in ("recall", "precision", "f1")},
        "macro": {m: d_agg_macro[m] for m in MACRO_METRICS},
    })

# Sort by macro-F1 mean descending
per_domain_report.sort(key=lambda x: x["macro"]["f1"]["mean"], reverse=True)

# Print per-domain table showing both micro-F1 and macro-F1 for quick comparison
print(f"  Sorted by macro-F1.  Full micro stats in JSON report.\n")
HDR = (f"  {'Domain':<25} {'N':>4}  {'Eval':>4}  "
       f"{'mi-Rec':>7}  {'mi-Pre':>7}  {'mi-F1':>7}  "
       f"{'ma-Rec':>7}  {'ma-Pre':>7}  {'ma-F1':>7}  "
       f"{'IPS':>7}  {'PLR':>7}")
print(HDR)
print(f"  {'-'*96}")
for dr in per_domain_report:
    mi = dr["micro"]
    ma = dr["macro"]
    print(
        f"  {dr['domain']:<25} {dr['sample_count']:>4}  {dr['evaluable_samples']:>4}  "
        f"{mi['recall']['mean']:>7.4f}  {mi['precision']['mean']:>7.4f}  {mi['f1']['mean']:>7.4f}  "
        f"{ma['recall']['mean']:>7.4f}  {ma['precision']['mean']:>7.4f}  {ma['f1']['mean']:>7.4f}  "
        f"{ma['ips']['mean']:>7.4f}  {ma['plr']['mean']:>7.4f}"
    )

############################################################
# SAVE REPORT
############################################################
report = {
    "meta": {
        "generated":     datetime.datetime.utcnow().isoformat() + "Z",
        "n_runs":        n_runs,
        "n_samples":     n_samples,
        "avg_skipped":   float(np.mean(skipped_counts)),
        "gliner_model":  meta_info["model"],
        "cache_created": meta_info["created"],
        "averaging": {
            "micro": "token bits pooled across all samples per run, then averaged across runs",
            "macro": "metric computed per sample, mean across samples per run, mean across runs",
            "note":  "IPS and PLR are inherently sample-level (scalar) so macro only",
        },
    },
    "per_run": {
        "micro": {
            m: [round(run_micro[r][m], 4)
                if not np.isnan(run_micro[r][m]) else None
                for r in range(n_runs)]
            for m in ("recall", "precision", "f1")
        },
        "macro": {
            m: [round(run_macro_mean[r][m], 4)
                if not np.isnan(run_macro_mean[r][m]) else None
                for r in range(n_runs)]
            for m in MACRO_METRICS
        },
    },
    "aggregate": {
        "micro": {m: agg_micro[m] for m in ("recall", "precision", "f1")},
        "macro": {m: agg_macro[m] for m in MACRO_METRICS},
    },
    "per_domain": per_domain_report,
}

with open(REPORT_FILE, "w") as f:
    json.dump(report, f, indent=2)

print(f"\n  Report saved -> {REPORT_FILE}")
print("Done.")